<a href="https://colab.research.google.com/github/RISHOBGHOSH/Pyspark/blob/main/Joins_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**22 Optimize Joins in Spark & Understand Bucketing for Faster joins |Sort Merge Join |Broad Cast Join**


In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Optimizing Joins")
    .master("local[*]")
    .config("spark.cores.max", 16)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [ ]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

Join Big and Small table - SortMerge vs BroadCast Join

In [ ]:
# Read EMP CSV data

_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/content/employee_records.csv")

emp.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [ ]:
# Read DEPT CSV data

_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load("/content/department_data.csv")

dept.show()

+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|department_id|     department_name|         description|                city|state|            country|
+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|            1|         Bryan-James|Optimized disinte...|        Melissaburgh|   FM|Trinidad and Tobago|
|            2|Smith, Craig and ...|Digitized empower...|          Morrisside|   DE|          Sri Lanka|
|            3|Pittman, Hess and...|Multi-channeled c...|         North David|   SC|       Turkmenistan|
|            4|Smith, Snyder and...|Reactive neutral ...|       Lake Jennifer|   TX|         Madagascar|
|            5|          Hardin Inc|Re-contextualized...|           Hayestown|   WA|               Fiji|
|            6|         Sanders LLC|Innovative multim...|         Phamchester|   TN|         Micronesia|
|            7|         Ward-Gordon|Progressive logis..

In [ ]:
# Join Datasets

df_joined = emp.join(dept, on="department_id", how="left_outer")

df_joined.show()

+-------------+-----------+---------+--------------------+----------+--------------------+--------------------+--------+--------------------+--------------------+------------+-----+-------------------+
|department_id| first_name|last_name|           job_title|       dob|               email|               phone|  salary|     department_name|         description|        city|state|            country|
+-------------+-----------+---------+--------------------+----------+--------------------+--------------------+--------+--------------------+--------------------+------------+-----+-------------------+
|            1|       John|   Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|         Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|
|            1|    Rachael|Rodriguez|         Media buyer|1966-12-02|griffinmary@examp...| +1-791-344-7586x548|544732.0|         Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad an

In [ ]:
df_joined.write.format("noop").mode("overwrite").save()

In [ ]:
df_joined.explain()


== Physical Plan ==
*(4) Project [department_id#7, first_name#0, last_name#1, job_title#2, dob#3, email#4, phone#5, salary#6, department_name#103, description#104, city#105, state#106, country#107]
+- *(4) SortMergeJoin [department_id#7], [department_id#102], LeftOuter
   :- *(1) Sort [department_id#7 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(department_id#7, 200), ENSURE_REQUIREMENTS, [plan_id=188]
   :     +- FileScan csv [first_name#0,last_name#1,job_title#2,dob#3,email#4,phone#5,salary#6,department_id#7] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- *(3) Sort [department_id#102 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(department_id#102, 200), ENSURE_REQUIREMENTS, [plan_id=201]
         +- *(2) Filter isnotnull(d

In [ ]:
# Join Datasets Broadcast
# Optimizing Big Table (emp) and Small Table (dept)
from pyspark.sql.functions import broadcast

df_joined = emp.join(broadcast(dept), on=emp.department_id==dept.department_id, how="left_outer")

In [ ]:
df_joined.write.format("noop").mode("overwrite").save()

**👉 NO-OP = Spark skips an unnecessary step because it has no effect.**

In [ ]:
df_joined.explain()


== Physical Plan ==
*(2) BroadcastHashJoin [department_id#7], [department_id#102], LeftOuter, BuildRight, false
:- FileScan csv [first_name#0,last_name#1,job_title#2,dob#3,email#4,phone#5,salary#6,department_id#7] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
+- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=280]
   +- *(1) Filter isnotnull(department_id#102)
      +- FileScan csv [department_id#102,department_name#103,description#104,city#105,state#106,country#107] Batched: false, DataFilters: [isnotnull(department_id#102)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/department_data.csv], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<department_

Join Big and Big table - SortMerge without Buckets

In [ ]:
# Read Sales data

sales_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"

sales = spark.read.format("csv").schema(sales_schema).option("header", True).load("/content/sales.csv")

In [ ]:
sales.show()

+--------------------+----------+-----------+--------------------+-------+----------+
|       transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+--------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24T19:00:...|1995601912| 2077350195|Walgreen       11-25| 197.23| 216510442|
|2017-11-24T19:00:...|1734117021|  644879053|unkn    ppd id: 7...|   8.58| 930259917|
|2017-11-24T19:00:...|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24T19:00:...|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24T19:00:...|1734117089| 1898522855| Target        11-25|  66.33|1855530529|
|2017-11-24T19:00:...|1734117117|  997626433|Sears  ppd id: 85...| 298.87| 957346984|
|2017-11-24T19:00:...|1734117123| 1953761884|unkn   ppd id: 15...|  19.55|  45522086|
|2017-11-24T19:00:...|1734117152| 1429095612|Ikea     arc id: ...|   9.39|1268541279|
|2017-11-24T19:00:...|1734117153|  847200066|unkn     

In [ ]:
# Read City data

city_schema = "city_id string, city string, state string, state_abv string, country string"

city = spark.read.format("csv").schema(city_schema).option("header", True).load("/content/cities.csv")

In [ ]:
# Join Data

df_sales_joined = sales.join(city, on=sales.city_id==city.city_id, how="left_outer")

In [ ]:
df_sales_joined.write.format("noop").mode("overwrite").save()


In [ ]:
# Explain Plan

df_sales_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [city_id#448], [city_id#503], LeftOuter
:- *(1) Sort [city_id#448 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(city_id#448, 200), ENSURE_REQUIREMENTS, [plan_id=570]
:     +- FileScan csv [transacted_at#443,trx_id#444,retailer_id#445,description#446,amount#447,city_id#448] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/sales.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit...
+- *(3) Sort [city_id#503 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(city_id#503, 200), ENSURE_REQUIREMENTS, [plan_id=582]
      +- *(2) Filter isnotnull(city_id#503)
         +- FileScan csv [city_id#503,city#504,state#505,state_abv#506,country#507] Batched: false, DataFilters: [isnotnull(city_id#503)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/cities.csv], PartitionFil

Write Sales and City data in Buckets


In [ ]:
# Write Sales data in Buckets

sales.write.format("csv").mode("overwrite").bucketBy(4, "city_id").option("header", True).option("path", "/content/sample_data/sales_bucket.csv").saveAsTable("sales_bucket")

In [ ]:
# Write City data in Buckets

city.write.format("csv").mode("overwrite").bucketBy(4, "city_id").option("header", True).option("path", "/content/sample_data/city_bucket.csv").saveAsTable("city_bucket")

In [ ]:
# Check tables

spark.sql("show tables in default").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default| city_bucket|      false|
|  default|sales_bucket|      false|
+---------+------------+-----------+



Join Sales and City data - SortMerge with Bucket


In [ ]:
# Read Sales table

sales_bucket = spark.read.table("sales_bucket")

In [ ]:
# Read City table

city_bucket = spark.read.table("city_bucket")

In [ ]:
# Join datasets

df_joined_bucket = sales_bucket.join(city_bucket, on=sales_bucket.city_id==city_bucket.city_id, how="left_outer")


In [ ]:
# Write dataset

df_joined_bucket.write.format("noop").mode("overwrite").save()

In [ ]:
df_joined_bucket.explain()


== Physical Plan ==
*(3) SortMergeJoin [city_id#620], [city_id#627], LeftOuter
:- *(1) Sort [city_id#620 ASC NULLS FIRST], false, 0
:  +- FileScan csv spark_catalog.default.sales_bucket[transacted_at#615,trx_id#616,retailer_id#617,description#618,amount#619,city_id#620] Batched: false, Bucketed: true, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/sample_data/sales_bucket.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit..., SelectedBucketsCount: 4 out of 4
+- *(2) Sort [city_id#627 ASC NULLS FIRST], false, 0
   +- *(2) Filter isnotnull(city_id#627)
      +- FileScan csv spark_catalog.default.city_bucket[city_id#627,city#628,state#629,state_abv#630,country#631] Batched: false, Bucketed: true, DataFilters: [isnotnull(city_id#627)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/sample_data/city_bucket.csv], PartitionFilters: [], Pu

**Points To Note For Optimization**

1.⁠ ⁠Joining Column different than Bucket Column, Same Bucket Size - Shuffle on Both table

2.⁠ ⁠Joining Column Same, One table in Bucket - Shuffle on non Bucket table

3.⁠ ⁠Joining Column Same, Different Bucket Size - Shuffle on Smaller Bucket Side

4.⁠ ⁠Joining Column Same,
Same Bucket Size - No Shuffle (Faster Join)


**Some More to Optimize**

1.⁠ ⁠So its very importatant to choose correct Bucket column and Bucket Size

2.⁠ ⁠Decide effectively on number of Buckets, as too many buckets with not enough data can lead to Small file issue.

3.⁠ ⁠Datasets are Small - you can prefer Shuffle Hash Join

⭐ 1. What is Dynamic Resource Allocation in Spark?

Dynamic Resource Allocation (DRA) allows Spark to automatically increase or decrease the number of executors during runtime based on workload.

Spark adjusts executors dynamically depending on:

Number of pending tasks

Data volume

Load on cluster

👉 If workload increases → Spark adds executors
👉 If workload decreases → Spark removes idle executors

This helps optimize cost, speed, and cluster utilization.

⭐ 2. How does Dynamic Resource Allocation work internally?

DRA requires:

Executor allocation manager

Shuffle service running on each node

Cluster manager (YARN, Kubernetes, Standalone)

Executors get added & removed without killing shuffle data.

⭐ 3. How to Configure Dynamic Resource Allocation in Spark

Add these configs in Spark:

spark.dynamicAllocation.enabled true
spark.dynamicAllocation.minExecutors 2
spark.dynamicAllocation.maxExecutors 50
spark.dynamicAllocation.initialExecutors 5
spark.dynamicAllocation.executorIdleTimeout 60s
spark.shuffle.service.enabled true

🔑 Required setting:
spark.shuffle.service.enabled true


Without this, Spark cannot remove executors safely.

You can set DRA via:

spark-defaults.conf

SparkSession builder

Databricks cluster configs (Classic only)

🧪 Example in SparkSession:

spark = SparkSession.builder \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "2") \
    .config("spark.dynamicAllocation.maxExecutors", "20") \
    .config("spark.shuffle.service.enabled", "true") \
    .getOrCreate()

⭐ 4. Static vs Dynamic Resource Allocation
Feature	Static Allocation	Dynamic Allocation
Executors	Fixed (e.g. always 10)	Grows/shrinks based on load
Cost	Higher	Lower (pay only for needed executors)
Flexibility	Rigid	Flexible
Performance	May underperform or overprovision	More optimal
Shuffle daemon	Not required	Required
Ideal for	Predictable workloads	Variable workloads
⭐ 5. How is Dynamic Resource Allocation different from Databricks Autoscaling?

This is very important — most people confuse both.

🔥 Dynamic Resource Allocation (Spark feature)
Happens within Spark application

Adds/removes executors only

Driver stays the same

Works inside an existing cluster

Controlled by Spark configs

Requires shuffle service

Example:

Your job increases from 5 → 20 executors.

🔥 Databricks Autoscaling (Cluster-level feature)
Happens at the cluster level

Adds/removes worker nodes (VMs)

Executors come with new nodes

Cluster grows/shrinks

Managed by Databricks

No shuffle service needed

More powerful than Spark DRA

Example:

Cluster scales from 5 → 20 nodes.

🧠 Key Differences (Interview-Ready)
Feature	Spark Dynamic Allocation	Databricks Autoscaling
Scaling level	Executor level	Cluster (VM) level
Managed by	Spark runtime	Databricks control plane
Adds	Executors	Worker nodes + executors
Removes	Executors	Worker nodes
Shuffle service needed	Yes	No
Works with	YARN, K8s, Standalone	Databricks cluster only
Purpose	Optimize Spark app resources	Optimize cluster cost & performance
🎯 Simple Analogy
Dynamic Resource Allocation:

Adjusting the number of workers at your office.

Databricks Autoscaling:

Renting additional office floors (hardware) or releasing them.

⭐ Interview One-Liner

Dynamic Resource Allocation changes only Spark executors based on workload, while Databricks Autoscaling changes the number of worker nodes in the cluster. DRA is inside the Spark app; autoscaling is outside at the infrastructure level.

**24 Fix Skewness and Spillage with Salting in Spark | Salting Technique | How to identify Skewness**

In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Optimizing Skewness and Spillage")
    .master("local[*]")
    .config("spark.cores.max", 8)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [ ]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
# Read Employee data
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/content/employee_records.csv")

In [ ]:
# Read DEPT CSV data
_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load("/content/department_data.csv")

In [ ]:
# Join Datasets

df_joined = emp.join(dept, on=emp.department_id==dept.department_id, how="left_outer")

In [ ]:
df_joined.write.format("noop").mode("overwrite").save()


In [ ]:
#Explain Plan

df_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [department_id#702], [department_id#723], LeftOuter
:- *(1) Sort [department_id#702 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(department_id#702, 200), ENSURE_REQUIREMENTS, [plan_id=815]
:     +- FileScan csv [first_name#695,last_name#696,job_title#697,dob#698,email#699,phone#700,salary#701,department_id#702] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
+- *(3) Sort [department_id#723 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(department_id#723, 200), ENSURE_REQUIREMENTS, [plan_id=827]
      +- *(2) Filter isnotnull(department_id#723)
         +- FileScan csv [department_id#723,department_name#724,description#725,city#726,state#727,country#728] Batched: false, DataFilters: [isnotnull(departm

What is Skewness?

Skewness is the uneven distribution of data across partitions or keys that causes certain Spark tasks to take much longer than others, leading to performance bottlenecks.

In [ ]:
# Check the partition details to understand distribution
from pyspark.sql.functions import spark_partition_id, count, lit

part_df = df_joined.withColumn("partition_num", spark_partition_id()).groupBy("partition_num").agg(count(lit(1)).alias("count"))

part_df.show()

+-------------+------+
|partition_num| count|
+-------------+------+
|          103|100417|
|          122| 99780|
|           43| 99451|
|          107| 99805|
|           49| 99706|
|           51|100248|
|          102|100214|
|           66|100210|
|          174|100155|
|           89|100014|
+-------------+------+



In [ ]:
# Verify Employee data based on department_id
from pyspark.sql.functions import count, lit, desc, col

emp.groupBy("department_id").agg(count(lit(1))).show()

+-------------+--------+
|department_id|count(1)|
+-------------+--------+
|            1|   99451|
|            6|   99706|
|            3|  100248|
|            5|  100210|
|            9|  100014|
|            4|  100214|
|            8|  100417|
|            7|   99805|
|           10|   99780|
|            2|  100155|
+-------------+--------+



In [ ]:
# Set shuffle partitions to a lesser number - 16 - but changed to 32 for better

spark.conf.set("spark.sql.shuffle.partitions", 32)

In [ ]:
# Let prepare the salt
import random
from pyspark.sql.functions import udf

# UDF to return a random number every time and add to Employee as salt
@udf
def salt_udf():
    return random.randint(0, 32)

# Salt Data Frame to add to department
salt_df = spark.range(0, 32)
salt_df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
| 12|
| 13|
| 14|
| 15|
| 16|
| 17|
| 18|
| 19|
+---+
only showing top 20 rows



Salting is a technique used to solve data skew in Spark by artificially adding randomness to skewed keys so that the data gets distributed more evenly across partitions.

In simple words:

👉 When one join key has too much data → you add a “salt” value to spread it across multiple partitions.

⭐ Why Do We Need Salting? (Problem)

Data Skew happens when:

One key has millions of rows

Other keys have very few

Salting is a technique to fix data skew in Spark.
When one key contains too many records, Spark assigns a random salt value to the key (e.g., appending 0–4), splitting the heavy key across multiple partitions.
Both tables are salted the same way, and the join is performed on the salted key.
This removes skew and improves performance by parallelizing the heavy key across multiple executors.

In [ ]:
# Salted Employee
from pyspark.sql.functions import lit, concat

salted_emp = emp.withColumn("salted_dept_id", concat("department_id", lit("_"), salt_udf()))

salted_emp.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|salted_dept_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|           8_1|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|          7_23|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|         10_11|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|          1_27|
|  Michelle|   Elliott|      Air cabin crew|1975

In [ ]:
# Salted Department - need to do cross join

salted_dept = dept.join(salt_df, how="cross").withColumn("salted_dept_id", concat("department_id", lit("_"), "id"))

salted_dept.where("department_id = 9").show()

+-------------+--------------------+--------------------+-----------+-----+-------+---+--------------+
|department_id|     department_name|         description|       city|state|country| id|salted_dept_id|
+-------------+--------------------+--------------------+-----------+-----+-------+---+--------------+
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   MN|  Italy|  0|           9_0|
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   MN|  Italy|  1|           9_1|
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   MN|  Italy|  2|           9_2|
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   MN|  Italy|  3|           9_3|
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   MN|  Italy|  4|           9_4|
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   MN|  Italy|  5|           9_5|
|            9|Mcmahon, Terrell ...|De-engineered hig...|Marychester|   M

In [ ]:
# Lets make the salted join now
salted_joined_df = salted_emp.join(salted_dept, on=salted_emp.salted_dept_id==salted_dept.salted_dept_id, how="left_outer")

In [ ]:
salted_joined_df.write.format("noop").mode("overwrite").save()


In [ ]:
# Check the partition details to understand distribution
from pyspark.sql.functions import spark_partition_id, count

part_df = salted_joined_df.withColumn("partition_num", spark_partition_id()).groupBy("partition_num").agg(count(lit(1)).alias("count")).orderBy("partition_num")

part_df.show()

+-------------+-----+
|partition_num|count|
+-------------+-----+
|            0|72847|
|            1|45447|
|            2|81874|
|            3|63796|
|            4|54512|
|            5|66833|
|            6|54723|
|            7|75776|
|            8|60573|
|            9|67017|
|           10|45282|
|           11|87629|
|           12|42241|
|           13|63414|
|           14|54439|
|           15|63597|
+-------------+-----+



In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("AQE in Spark")
    .master("local[*]")
    .config("spark.cores.max", 8)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [ ]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
# Read Employee data
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/content/employee_records.csv")

In [ ]:
# Read DEPT CSV data
_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load("/content/department_data.csv")

In [ ]:
# Join Datasets

df_joined = emp.join(dept, on=emp.department_id==dept.department_id, how="left_outer")


In [ ]:
df_joined.write.format("noop").mode("overwrite").save()


In [ ]:
#Explain Plan

df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [department_id#1990], [department_id#2011], LeftOuter
   :- Sort [department_id#1990 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(department_id#1990, 16), ENSURE_REQUIREMENTS, [plan_id=2632]
   :     +- FileScan csv [first_name#1983,last_name#1984,job_title#1985,dob#1986,email#1987,phone#1988,salary#1989,department_id#1990] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- Sort [department_id#2011 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(department_id#2011, 16), ENSURE_REQUIREMENTS, [plan_id=2633]
         +- Filter isnotnull(department_id#2011)
            +- FileScan csv [department_id#2011,department_name#2012,description#2013,city#2014,state#2015,

In [ ]:
df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [department_id#1990], [department_id#2011], LeftOuter, BuildRight, false
   :- FileScan csv [first_name#1983,last_name#1984,job_title#1985,dob#1986,email#1987,phone#1988,salary#1989,department_id#1990] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=3108]
      +- Filter isnotnull(department_id#2011)
         +- FileScan csv [department_id#2011,department_name#2012,description#2013,city#2014,state#2015,country#2016] Batched: false, DataFilters: [isnotnull(department_id#2011)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/department_data.csv], PartitionFilters:

In [ ]:
# Coalescing post-shuffle partitions - remove un-necessary shuffle partitions
# Skewed join optimization (balance partitions size) - join smaller partitions and split bigger partition

spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", True)

In [ ]:
# Fix partition sizes to avoid Skew

spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "8MB") #Default value: 64MB
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "10MB") #Default value: 256MB

In [ ]:
# Check the partition details to understand distribution
from pyspark.sql.functions import spark_partition_id, count, lit

part_df = df_joined.withColumn("partition_num", spark_partition_id()).groupBy("partition_num").agg(count(lit(1)).alias("count"))

part_df.show()

+-------------+-------+
|partition_num|  count|
+-------------+-------+
|            0|1000000|
+-------------+-------+



In [ ]:
# Converting sort-merge join to broadcast join

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10MB")

In [ ]:
# Join Datasets - without specifying specific broadcast table

df_joined = emp.join(dept, on=emp.department_id==dept.department_id, how="left_outer")

In [ ]:
df_joined.write.format("noop").mode("overwrite").save()


**26 Spark SQL, Hints, Spark Catalog and Metastore | Hints in Spark SQL Query | SQL functions & Joins**


In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark SQL")
    .master("local[*]")
    #.enableHiveSupport() - persist the MetaData in Hive Catalog
    .config("spark.sql.warehouse.dir", "/data/output/spark-warehouse")
    .getOrCreate()
)

spark

In [ ]:
# Read Employee data
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/content/employee_records.csv")

emp.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [ ]:
# Read DEPT CSV data
_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load("/content/department_data.csv")

dept.show()

+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|department_id|     department_name|         description|                city|state|            country|
+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|            1|         Bryan-James|Optimized disinte...|        Melissaburgh|   FM|Trinidad and Tobago|
|            2|Smith, Craig and ...|Digitized empower...|          Morrisside|   DE|          Sri Lanka|
|            3|Pittman, Hess and...|Multi-channeled c...|         North David|   SC|       Turkmenistan|
|            4|Smith, Snyder and...|Reactive neutral ...|       Lake Jennifer|   TX|         Madagascar|
|            5|          Hardin Inc|Re-contextualized...|           Hayestown|   WA|               Fiji|
|            6|         Sanders LLC|Innovative multim...|         Phamchester|   TN|         Micronesia|
|            7|         Ward-Gordon|Progressive logis..

In [ ]:
# Spark Catalog (Metadata) - in-memory/hive

spark.conf.get("spark.sql.catalogImplementation")

'hive'

🔥 Technical Definition (Interview-Ready)

A Catalog is a top-level namespace in Spark/Databricks that stores metadata about managed/unmanaged tables, schemas, files, views, and functions.

Metadata is information that describes the structure, meaning, and properties of data. It includes schemas, data types, columns, partitions, lineage, owners, and storage details. Metadata helps organize, govern, and optimize data processing in Spark and Databricks.

In [ ]:
spark.sql("show databases")

DataFrame[namespace: string]

In [ ]:
# Show databases
db = spark.sql("show databases")
db.show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [ ]:
spark.sql("show tables in default").show()


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [ ]:
# Register dataframes are temp views

emp.createOrReplaceTempView("emp_view")

dept.createOrReplaceTempView("dept_view")

#createGlobalTempView - will be available for other sessions
#createOrReplaceGlobalTempView
#createOrReplaceTempView
#createTempView



In [ ]:
spark.sql("show tables in default").show()

#temp views are only available until the session is active


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |dept_view|       true|
|         | emp_view|       true|
+---------+---------+-----------+



In [ ]:
# View data from table

spark.sql("""
    select * from emp_view
""")



DataFrame[first_name: string, last_name: string, job_title: string, dob: string, email: string, phone: string, salary: double, department_id: int]

In [ ]:
spark.sql("""
    select * from emp_view
    where department_id = 1
""").show()

+-----------+---------+--------------------+----------+--------------------+--------------------+--------+-------------+
| first_name|last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+-----------+---------+--------------------+----------+--------------------+--------------------+--------+-------------+
|       John|   Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|    Rachael|Rodriguez|         Media buyer|1966-12-02|griffinmary@examp...| +1-791-344-7586x548|544732.0|            1|
|Christopher| Callahan| Exhibition designer|1966-10-23| qwalter@example.com|001-947-745-3939x...|251057.0|            1|
|    Lindsey|   Huerta|Embryologist, cli...|1964-10-20|  psmith@example.net|   527.934.6665x1378|878257.0|            1|
|      David|   Harris|   Company secretary|1990-04-13|     nli@example.com|001-959-766-1180x...|249553.0|            1|
|      Brian|Hernandez|     Thea

In [ ]:
emp_filtered = spark.sql("""
    select * from emp_view
    where department_id = 1
""")
emp_filtered.show()

+-----------+---------+--------------------+----------+--------------------+--------------------+--------+-------------+
| first_name|last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+-----------+---------+--------------------+----------+--------------------+--------------------+--------+-------------+
|       John|   Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|    Rachael|Rodriguez|         Media buyer|1966-12-02|griffinmary@examp...| +1-791-344-7586x548|544732.0|            1|
|Christopher| Callahan| Exhibition designer|1966-10-23| qwalter@example.com|001-947-745-3939x...|251057.0|            1|
|    Lindsey|   Huerta|Embryologist, cli...|1964-10-20|  psmith@example.net|   527.934.6665x1378|878257.0|            1|
|      David|   Harris|   Company secretary|1990-04-13|     nli@example.com|001-959-766-1180x...|249553.0|            1|
|      Brian|Hernandez|     Thea

In [ ]:
# Create a new column dob_year and register as temp view

emp_temp = spark.sql("""
    select e.*, date_format(dob, 'yyyy') as dob_year from emp_view e
""")

Pyspark DataFrame API functions are available for Spark SQL by default , No need for any import

In [ ]:
emp_temp.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|dob_year|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|    1973|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|    1974|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|    1990|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|    1968|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)90

In [ ]:
emp_temp.createOrReplaceTempView("emp_temp_view")


In [ ]:
spark.sql("show tables in default").show()


+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|         |    dept_view|       true|
|         |emp_temp_view|       true|
|         |     emp_view|       true|
+---------+-------------+-----------+



In [ ]:
spark.sql("select * from emp_temp_view").show()


+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|dob_year|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|    1973|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|    1974|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|    1990|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|    1968|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)90

In [ ]:
# Join emp and dept - HINTs


spark.sql("""
    select e.* , d.department_name
    from emp_view e left outer join dept_view d
    on e.department_id = d.department_id
""")

DataFrame[first_name: string, last_name: string, job_title: string, dob: string, email: string, phone: string, salary: double, department_id: int, department_name: string]

In [ ]:
emp_join = spark.sql("""
    select e.* , d.department_name
    from emp_view e left outer join dept_view d
    on e.department_id = d.department_id
""")

emp_join.show()




+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|          Parker PLC|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|         Ward-Gordon|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|      Delgado-Keller|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|         Bryan-James|
|  Mic

In [ ]:
# Hints - in Spark SQL

emp_final_1 = spark.sql("""
    select /*+ SHUFFLE_MERGE(e) */
    e.* , d.department_name
    from emp_view e left outer join dept_view d
    on e.department_id = d.department_id
""").show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|         Bryan-James|
|   Rachael| Rodriguez|         Media buyer|1966-12-02|griffinmary@examp...| +1-791-344-7586x548|544732.0|            1|         Bryan-James|
|   Phillip|      Sims|Trading standards...|1979-04-03|jasonmarquez@exam...|        697.201.2204|326537.0|            1|         Bryan-James|
|       Amy|    Miller|Insurance account...|1997-12-01|dcollins@example.net|        605-471-4748|370531.0|            1|         Bryan-James|
|     

In [ ]:
emp_final = spark.sql("""
    select /*+ BROADCAST(d) */
    e.* , d.department_name
    from emp_view e left outer join dept_view d
    on e.department_id = d.department_id
""")

In [ ]:
# Show emp data

emp_final.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|          Parker PLC|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|         Ward-Gordon|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|      Delgado-Keller|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|         Bryan-James|
|  Mic

In [ ]:
# Write the data as Table

emp_final.write.format("parquet").saveAsTable("emp_final")

In [ ]:
# Read the data from Table

emp_new = spark.sql("select * from emp_final")

In [ ]:
spark.sql("show tables in default").show()


+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|  default|    emp_final|      false|
|         |    dept_view|       true|
|         |emp_temp_view|       true|
|         |     emp_view|       true|
+---------+-------------+-----------+



In [ ]:
# Read the data from Table

emp_new_1 = spark.read.table("emp_final").show()
# emp_new_1.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|          Parker PLC|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|         Ward-Gordon|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|      Delgado-Keller|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|         Bryan-James|
|  Mic

In [ ]:
# Read the data from Table
# Because we will have the Data in Catalog - in memory - which will lost onces we restart the Kernel
# we can persist Metastore with HIVE catalog

emp_new = spark.sql("select * from emp_final")

In [ ]:
emp_new.show()


+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|          Parker PLC|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|         Ward-Gordon|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|      Delgado-Keller|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|         Bryan-James|
|  Mic

In [ ]:
# we can persist Metastore with HIVE catalog

# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark SQL")
    .master("local[*]")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "/data/output/spark-warehouse")
    .getOrCreate()
)

spark

In [ ]:
# Show details of metadata

spark.sql("describe emp_final").show()

+---------------+---------+-------+
|       col_name|data_type|comment|
+---------------+---------+-------+
|     first_name|   string|   NULL|
|      last_name|   string|   NULL|
|      job_title|   string|   NULL|
|            dob|   string|   NULL|
|          email|   string|   NULL|
|          phone|   string|   NULL|
|         salary|   double|   NULL|
|  department_id|      int|   NULL|
|department_name|   string|   NULL|
+---------------+---------+-------+



In [ ]:
# Show details of metadata - extended

spark.sql("describe extended emp_final").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          first_name|              string|   NULL|
|           last_name|              string|   NULL|
|           job_title|              string|   NULL|
|                 dob|              string|   NULL|
|               email|              string|   NULL|
|               phone|              string|   NULL|
|              salary|              double|   NULL|
|       department_id|                 int|   NULL|
|     department_name|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|             default|       |
|               Table|           emp_final|       |
|               Owner|                root|       |
|        Created Time|Tue Dec 09 15:19:...|       |
|         La

27 Read and Write from Azure Cosmos DB using Spark | E2E Cosmos DB setup | NoSQL vs SQL Databases

In [ ]:
 # Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Read and Write using Cosmos DB")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'com.azure.cosmos.spark:azure-cosmos-spark_3-3_2-12:4.15.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
# Set configuration settings to connect to Cosmos DB

# in Azure cosmos DB under Keys

# in the last part of the video - it was give a way to hid the Keys

config = {
  "spark.cosmos.accountEndpoint": "<cosmos-db-endpoint>",
  "spark.cosmos.accountKey": "<secret-key>",
  "spark.cosmos.database": "easewithdata",
  "spark.cosmos.container": "device-data"
}

In [ ]:
# Read data from Cosmos DB

df = (
    spark.read.format("cosmos.oltp")
    .options(**config)
    .option("spark.cosmos.read.inferSchema.enabled", "true")
    .load()

)

In [ ]:
df.printSchema()

In [ ]:
df.show()

In [ ]:
# Write data to Cosmos DB

df_read = spark.read.json("datasets/devices/device_03.json")

In [ ]:
df_read.show()

In [ ]:
# Write data to Cosmos DB
from pyspark.sql.functions import col

df_read.withColumn("id", col("eventId")).write \
    .format("cosmos.oltp") \
    .options(**config) \
    .option("spark.cosmos.write.strategy", "ItemDelete") \
    .mode("APPEND") \
    .save()

28 Get Started with Delta Lake using Databricks | Benefits and Features of Delta Lake | Time Travel

In [ ]:
spark


In [ ]:
# Default Catalog for Databricks
spark.conf.get("spark.sql.catalogImplementation")

'in-memory'

watch the Video - no databricks code written

https://www.youtube.com/watch?v=84HOVF8Vn3I&list=PL2IsFZBGM_IHCl9zhRVC1EXTomkEp_1zm&index=30

30 Data Skipping and Z-Ordering in Delta Lake Tables | Optimize & Data Compaction Delta Lake Tables

Check the Git hub - https://github.com/subhamkharwal/pyspark-zero-to-hero/blob/master/25_delta_lake_optimization_and_z_ordering.ipynb

In [ ]:
# Dataset
display(dbutils.fs.ls("/databricks-datasets/definitive-guide/data/retail-data/all/"))


In [ ]:
%fs head dbfs:/data/input/sales/sales.csv


In [ ]:
# Write the data in form of delta table
df = spark.read.csv(path="/data/input/sales/sales.csv", inferSchema=True, header=True)
df.repartition(16).write.format("delta").mode("overwrite").partitionBy("country").option("path", "/data/output/sales_delta_partitioned/").saveAsTable("sales_delta_partitioned")


In [ ]:
%sql
select * from sales_delta

In [ ]:
# Data at delta location
display(dbutils.fs.ls("/data/output/sales_delta/"))


In [ ]:
%sql
select * from sales_delta where InvoiceNo = '576394'

In [ ]:
# _metadata.file_name is hidden and will store the source metadata
%sql
select min(invoiceno), max(invoiceno), _metadata.file_name from sales_delta
group by _metadata.file_name
order by min(invoiceno)



In [ ]:
# this configiration is used to control the file compaction
# max size using OPTIMIZE command

spark.conf.set("spark.databricks.delta.optimize.maxFileSize", 64*1024*8)


In [ ]:
%sql
OPTIMIZE sales_delta ZORDER BY (InvoiceNo)

# optimization of the high cardinality column in delta lake
# here the delta has got all the sequence number and arranged it in sequence ordered and then distributed in part files

In [ ]:
%sql
select min(invoiceno), max(invoiceno), _metadata.file_name from sales_delta
group by _metadata.file_name
order by min(invoiceno)

In [ ]:
%sql
select * from sales_delta where InvoiceNo = '576394'

# now this code is take less time - as its only reading one parquet file - becoz we have Zordered the Delta lake

In [ ]:
%sql
OPTIMIZE sales_delta ZORDER BY (InvoiceNo)


In [ ]:
# Multi_dimentionsal

%sql
OPTIMIZE sales_delta ZORDER BY (Country , InvoiceNo)

In [ ]:
%sql
select country,min(invoiceno), max(invoiceno), _metadata.file_name from sales_delta
group by country,_metadata.file_name
order by country,min(invoiceno)

In [ ]:
%sql
select * from sales_delta where InvoiceNo = '576394'

# here the spark clusted will be reading 2 files becoz we added country in Z-order

In [ ]:
%sql
select * from sales_delta where InvoiceNo = '576394' and country = 'Australia'

# here the spark clusted will be reading 1 files becoz we added country in Z-order as we have added the filter

In [ ]:
# better way to it
# we will partition the table based on country and and Z-order with Invoice_no
# so that we dont have to Z-order the order data with counrty now

display(dbutils.fs.ls("/data/output/sales_delta_partitioned/"))




In [ ]:
# show the data

display(dbutils.fs.ls("/data/output/sales_delta_partitioned/Country=Australia/"))


In [ ]:
%sql
select * from sales_delta_partitioned where InvoiceNo = '576394' and country = 'Australia'

# no of file read was 16 , and file read partition was 1

In [ ]:
%sql
OPTIMIZE sales_delta_partitioned where country = 'Australia' ZORDER BY (InvoiceNo)

# no of file read was 1 , and file read partition was 1

# this data skipping at it best

31 Delta Tables - Deletion Vectors and Liquid Clustering | Optimize Delta Tables | Delta Clustering

code are there in DataBricks

32 Spark Memory Management | Why OOM Errors in Spark | Spark Unified Memory | Storage/Execution Mem

In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark Memory Management")
    .master("local[*]")
    .config("spark.cores.max", 8)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512MB")
    .getOrCreate()
)

spark

In [ ]:
# JVM On-Heap Usable memory (89% of executor memory)
512 * 0.89


455.68

In [ ]:
# Subtracting Reserve Memory (300MB)
455.68 - 300

155.68

In [ ]:
# Total Spark Memory (Unified Memory - Storage + Execution Memory) (60% default) spark.memory.fraction = 0.6

155.68 * 0.6

93.408

In [ ]:
# User / Undefined Memory (Not controlled by Spark) (remaining 40% default)

155.68 * 0.4



62.272000000000006

Go to spark UI and under Executors verify the Unified Storage Memory

In [ ]:
# Storage Memory (spark.memory.storageFraction = 0.5)
93.408 * 0.5


46.704

In [ ]:
# Execution Memory

93.408 * 0.5


46.704

In [ ]:
# Execution Memory per core
46.704 / 4

11.676

Out Of Memory Error Demo on Executors

In [ ]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
%%sh
ls -ltrh /data/datasets/oom_example/


In [ ]:
# Read file

df_1 = spark.read.format("text").load("/data/datasets/oom_example/text_file_singleline_xs.txt")

# df_2 = spark.read.format("text").option("wholetext",True).load("/data/datasets/oom_example/text_file_xs.txt")
# df_2 will again read the whole files as 1 single line and again it will explode and will result OOM

In [ ]:
df = spark.read.format("text").load("/data/datasets/oom_example/text_file_xs.txt")

# use this and not the above code - becoz above has only 1 single line as a record

In [ ]:
# Cache data
df.cache().count()

In [ ]:
df.printSchema()

Go to Spark UI in storage and Excutors - check teh storage memory

In [ ]:
# Explode data to count words
from pyspark.sql.functions import lower,explode, split , lit

df_final = (
    df.withColumn("value",lower("value"))
    .withColumn("splitted_val",split("value" ," ")) # ['is' , 'an']
    .withColumn("exploded_val",explode("splitted_val"))
    .drop("splitted_val" , "value")
    .groupBy("exploded_val")
    .agg(count(lit(1)).alias("cnt"))
)

In [ ]:
df_final.show()

Go to SPARK UI and under Jobs and see the Errors

In [ ]:
# Write with noop format for simulation
df_final.write.format("noop").mode("overwrite").save()

# used for benchmarking
# looks like we are writing the data in reality we are not